# Build a Scalable Multi-Agent AI System with Flyte

<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/multi-agent-workflows/tutorial_planner_agent.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Learn to build a planner agent system with intelligent routing, parallel execution, and automatic context passing between agents.

**Pattern:** Planner analyzes request → Creates dependency graph → Orchestrator executes in parallel waves

```
User: "Calculate 2+3 and 5+6, then add results"
  ↓
Planner: [Step 0: 2+3 (no deps), Step 1: 5+6 (no deps), Step 2: add (deps: 0,1)]
  ↓
Orchestrator: Wave 1 [0,1 parallel] → Wave 2 [2 with context]
```

**Key concepts:** Dynamic DAGs, dependency-aware parallelism, automatic context injection.

---

## Setup

Project structure: `agents/`, `tools/`, `workflows/`, `utils/`

**Configuration:** Shared Flyte environment with Docker image + secrets. Agents inherit this config but can override for custom resources.

In [1]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/multi-agent-workflows/
    !uv pip install -r requirements.txt
    
# this is just for viewing the code files within the notebook
from utils.file_viewer import view_file

In [2]:
view_file("requirements.txt")

In [3]:
view_file("config.py")

## Connect to Flyte Cluster
You can skip this step if you're not using a Flyte cluster and only want to run the examples locally.

Flyte gives you...

- If you don't have a Flyte cluster you can request demo access by filling out the form [here](https://flyte.org/).
- If you already have a Flyte cluster, you can connect to it by setting your endpoint in the Flyte configuration.


In [27]:
!flyte create config \
    --endpoint tryv2.hosted.unionai.cloud \
    --auth-type headless\
    --builder remote \
    --domain development \
    --project flytesnacks

Overwrite [/Users/sageelliott/Documents/gitrepos/workshops/tutorials/multi-agent-workflows/.flyte/config.yaml]? [y/N]: ^C


You can now adjust the configuration by modifying the `.flyte/config.yaml` file.

In [4]:
view_file(".flyte/config.yaml")

## Set your API Key(s)

The project is setup to read in secrets from a `.env` file.

You can create this file in the root of this tutorial `tutorials/multi-agent-workflows` and add your API keys there.

But if you prefer to just enter a key once in this notebook you can run the cell below:

In [ ]:
# Skip if API key is already set in .env or environment
import os
from getpass import getpass

os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')

To run on the remote Flyte cluster, add the API keys as secrets. 

You can skip this step if you're not using a Flyte cluster and only want to run the examples locally.


In [ ]:
# run this and enter your API key as the input
!flyte create secret OPENAI_API_KEY

## Run the Agent

At this point you should be setup to run the planner agent. 

I suggest giving it a try before we walk through the code in the next section.

**Run locally:**

In [80]:
!python -m workflows.planner --request "Do these tasks: calculate 2+2, 3*3, 4/2, 5-1, calculate 6 factorial, count words in 'one two three', count words in 'four five six', count characters in 'hello', count characters in 'world', and get weather in Boston. Then add up all the numbers you found." --local

Running workflow LOCALLY with flyte.init()

=== Planner Agent Workflow ===
Request: Do these tasks: calculate 2+2, 3*3, 4/2, 5-1, calculate 6 factorial, count words in 'one two three', count words in 'four five six', count characters in 'hello', count characters in 'world', and get weather in Boston. Then add up all the numbers you found.

[798fb782-44af-475e-9683-7e1d8577f900][dpg7lxcan0i4i3vhcyil27lo0] [Orchestrator] User request: Do these tasks: calculate 2+2, 3*3, 4/2, 5-1, calculate 6 factorial, count words in 'one two three', count words in 'four five six', count characters in 'hello', count characters in 'world', and get weather in Boston. Then add up all the numbers you found.
[798fb782-44af-475e-9683-7e1d8577f900][dpg7lxcan0i4i3vhcyil27lo0] [Orchestrator] Step 1: Calling planner agent...
[Planner Agent] Processing request: Do these tasks: calculate 2+2, 3*3, 4/2, 5-1, calculate 6 factorial, count words in 'one two three', count words in 'four five six', count characters in 'he

**Run on the remote Flyte cluster:**

The first time running the agent a container image will be built and pushed to the Flyte cluster.

This may take some time depending on the size of your dependencies.

In [ ]:
!python -m workflows.planner --request "Do these tasks: calculate 2+2, 3*3, 4/2, 5-1, calculate 6 factorial, count words in 'one two three', count words in 'four five six', count characters in 'hello', count characters in 'world', and get weather in Boston. Then add up all the numbers and the temp from weather."

Running workflow REMOTELY with flyte.init_from_config()

=== Planner Agent Workflow ===
Request: Do these tasks: calculate 2+2, 3*3, 4/2, 5-1, calculate 6 factorial, count words in 'one two three', count words in 'four five six', count characters in 'hello', count characters in 'world', and get weather in Boston. Then add up all the numbers and the temp from weather.

16:19:22.518752 WARNING  remote_builder.py:95 -  Image                          
                         356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo
                         :flyte-a6e80122ad3ee2d24341681f57f3b7a8 found. Skip    
                         building.                                              
16:19:22.524018 WARNING  _deploy.py:376 -  Built Image for environment base_env,
                         image:                                                 
                         356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo
                         :flyte-a6e80122ad3ee2d24341681f57f3b7

In [83]:
!python -m workflows.planner --request "Research the top 3 programming languages of 2024, write a Python script that generates a comparison chart as ASCII art, then count the words in the output"

Running workflow REMOTELY with flyte.init_from_config()

=== Planner Agent Workflow ===
Request: Research the top 3 programming languages of 2024, write a Python script that generates a comparison chart as ASCII art, then count the words in the output

17:13:54.028145 WARNING  remote_builder.py:95 -  Image                          
                         356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo
                         :flyte-a6e80122ad3ee2d24341681f57f3b7a8 found. Skip    
                         building.                                              
17:13:54.029687 WARNING  _deploy.py:376 -  Built Image for environment base_env,
                         image:                                                 
                         356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo
                         :flyte-a6e80122ad3ee2d24341681f57f3b7a8                

Execution: r2bwkrktxjgdz66ztst2
URL: https://demo.hosted.unionai.cloud/v2/domain/development/proje

In [84]:
!python -m workflows.planner --request "Find the population of Tokyo and New York, then calculate what percentage Tokyo's populationis of New York's"

Running workflow REMOTELY with flyte.init_from_config()

=== Planner Agent Workflow ===
Request: Find the population of Tokyo and New York, then calculate what percentage Tokyo's populationis of New York's

17:15:13.374419 WARNING  remote_builder.py:95 -  Image                          
                         356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo
                         :flyte-a6e80122ad3ee2d24341681f57f3b7a8 found. Skip    
                         building.                                              
17:15:13.375959 WARNING  _deploy.py:376 -  Built Image for environment base_env,
                         image:                                                 
                         356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo
                         :flyte-a6e80122ad3ee2d24341681f57f3b7a8                

Execution: r8lqfv6f8b7kpwlqt8g5
URL: https://demo.hosted.unionai.cloud/v2/domain/development/project/flytesnacks/runs/r8lqfv6f8b7kpwlqt8g5



# Code Walkthrough

Let's walk through the code to understand how the planner agent is structured and how it works.

We'll cover all the key file types, but you can explore all the agents and tools in their respective folders.

## Infrastructure

**Decorators** - Registration system for agents and tools. 

This allows for easy addition and management of new agents and tools within the workflow.

In [5]:
view_file("utils/decorators.py")

**Plan Executor** - Takes LLM-generated JSON tool plans and executes them. Uses "previous" keyword for chaining results between tool calls.

This is responsible for executing the plans generated by the LLM (planner) and managing the flow of data between different tools and agents.

In [6]:
view_file("utils/plan_executor.py")

---

## Tools

Tools are async functions that agents call to perform actions. Each agent has its own toolset.

**Pattern:**
```python
@tool(agent="math")
@flyte.trace
async def add(a: Numeric, b: Numeric) -> float:
    return float(a) + float(b)
```

**Key features:**
- Type annotations for input/output validation
- `@flyte.trace` for observability & durability within Flyte platform
- Registered per-agent for isolation
- Async enables parallel execution

**Toolsets available in this tutorial:** math (arithmetic), string (text analysis), web_search (DuckDuckGo + fetch), code (Python execution)

See them in the `\tools` directory.

### Math Tools

Let's explore the math tools available in this tutorial.

In [7]:
view_file("tools/math_tools.py")

### Web Search Tools

Let's explore the web search tools available in this tutorial.


In [8]:
view_file("tools/web_search_tools.py")


---

## Agents

Specialist agents that use tools to solve domain-specific tasks.

**Pattern:**
```python
@agent("math")
@env.task
async def math_agent(task: str) -> MathAgentResult:
    # 1. Build prompt with available tools
    # 2. Ask LLM for tool execution plan
    # 3. Execute plan via execute_tool_plan()
    # 4. Return structured result (dataclass)
```

**Design:**
- `@env.task` = Flyte containerization (scalable, isolated, durable, observable)
- `@agent()` = Registration (enables dynamic routing)
- Dataclass output = Type-safe, serializable
- Async = Parallel tool execution

### Math Agent

Let's explore the math agent available in this tutorial.

See others in the `\agents` directory.

In [29]:
view_file("agents/math_agent.py")

### Web Search Agent

Let's explore the web search agent available in this tutorial.

See others in the `\agents` directory.

In [9]:
view_file("agents/web_search_agent.py")


### Planner Agent - The Task Coordinator

This agent is responsible for understanding the user's request, breaking it down into smaller tasks, and determining the best order to execute those tasks while considering any dependencies between them to manage parallelism.

**Unique:** Coordinates other agents instead of using tools.

**What it does:**
1. Analyzes user request
2. Identifies which agents are needed
3. Determines dependencies (`dependencies=[0, 1]` means step waits for 0 and 1)
4. Returns execution plan

**Example:**
```
"Calculate 2+3 and 5+6, then add results"
→ [{agent: "math", task: "2+3", dependencies: []},
   {agent: "math", task: "5+6", dependencies: []},
   {agent: "math", task: "add results", dependencies: [0,1]}]
```

**Key:** Few-shot prompting teaches the LLM about dependencies → enables dynamic parallelization.

In [10]:
view_file("agents/planner_agent.py")

---

## Orchestrator - The Agentic Workflow

Executes plans with dependency-aware parallelism.

**Flow:**
1. Get plan from planner
2. Loop: Find steps with satisfied dependencies → Execute in parallel → Mark complete
3. For dependent steps: Inject previous results via `build_task_with_context()`

**Context passing:**
```python
# Step 2 depends on steps 0 and 1
task = """
RESULTS FROM PREVIOUS STEPS:
  - Step 0 (math): 5
  - Step 1 (math): 11

YOUR TASK:
Add the results
"""
```

**Key features:** Automatic parallelization (`asyncio.gather`), result propagation, circular dependency detection.

In [11]:

view_file("workflows/planner.py")

---

## Running the Workflow

**Local (development):**
```bash
python -m workflows.planner --local --request "your request"
```
In-process execution, fast iteration.

**Remote (production):**
```bash
python -m workflows.planner --request "your request"  
```
Distributed Flyte cluster, scalable and observable.

**Try these:**
- Simple: `"Calculate 5 factorial"`
- Parallel: `"Calculate 2+3 and 5+6, then add results"`
- Cross-agent: `"Search France's GDP, calculate 5% of it"`

In [33]:
!python -m workflows.planner --request "Calculate 10 times 5 and count words in 'Hello World', then multiply the word count by the calculation result"

Running workflow REMOTELY with flyte.init_from_config()

=== Planner Agent Workflow ===
Request: Calculate 10 times 5 and count words in 'Hello World', then multiply the word count by the calculation result

15:40:44.947010 WARNING  remote_builder.py:95 -  Image                          
                         356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo
                         :flyte-a6e80122ad3ee2d24341681f57f3b7a8 found. Skip    
                         building.                                              
15:40:44.949040 WARNING  _deploy.py:376 -  Built Image for environment base_env,
                         image:                                                 
                         356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo
                         :flyte-a6e80122ad3ee2d24341681f57f3b7a8                

Execution: rgzf2wbq4h69r8hpq9sm
URL: https://demo.hosted.unionai.cloud/v2/domain/development/project/flytesnacks/runs/rgzf2wbq4h69r8hpq9sm
Clic

In [31]:
!python -m workflows.planner --request "Search for France's GDP in 2023, then calculate 5% of that value"

Running workflow REMOTELY with flyte.init_from_config()

=== Planner Agent Workflow ===
Request: Search for France's GDP in 2023, then calculate 5% of that value

15:12:44.142693 WARNING  remote_builder.py:95 -  Image                          
                         356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo
                         :flyte-a6e80122ad3ee2d24341681f57f3b7a8 found. Skip    
                         building.                                              
15:12:44.144539 WARNING  _deploy.py:376 -  Built Image for environment base_env,
                         image:                                                 
                         356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo
                         :flyte-a6e80122ad3ee2d24341681f57f3b7a8                

Execution: rtq54xdmfk4m5mm4jzmd
URL: https://demo.hosted.unionai.cloud/v2/domain/development/project/flytesnacks/runs/rtq54xdmfk4m5mm4jzmd
Click the link above to view execution details in

In [20]:
!!python -m workflows.planner --request "Calculate 10 times 5 and count words in 'Hello World', then multiply the word count by the calculation result" --local

['Running workflow LOCALLY with flyte.init()',
 '',
 '=== Dynamic Multi-Agent Workflow ===',
 "Request: Calculate 10 times 5 and count words in 'Hello World', then multiply the word count by the calculation result",
 '',
 "[Orchestrator] User request: Calculate 10 times 5 and count words in 'Hello World', then multiply the word count by the calculation result",
 '[Orchestrator] Step 1: Calling planner agent...',
 "[Planner Agent] Processing request: Calculate 10 times 5 and count words in 'Hello World', then multiply the word count by the calculation result",
 '[Planner Agent] Raw result: {\'steps\': [{\'agent\': \'math\', \'task\': \'Calculate 10 times 5\', \'dependencies\': []}, {\'agent\': \'string\', \'task\': "Count words in \'Hello World\'", \'dependencies\': []}, {\'agent\': \'math\', \'task\': \'Multiply the result from step 0 by the word count from step 1\', \'dependencies\': [0, 1]}]}',
 '[Planner Agent] Plan has 3 step(s)',
 '[Planner Agent]   Step 0: [math] Calculate 10 tim

---

## Key Takeaways

**Architecture:**
- **Modular:** Agent + tools pattern, easy to extend
- **Parallel:** Automatic fanout for independent tasks
- **Smart:** LLM-generated dependency graphs
- **Production:** Type-safe, observable, scalable with Flyte

**What makes this powerful:**
Dynamic DAG generation - the LLM creates the execution plan from natural language, Flyte executes it efficiently.

**Next steps:**
1. Add your own agents (`agents/my_agent.py`)
2. Create tools (`tools/my_tools.py`)
3. Import in workflows - automatic registration handles the rest

Experiment with different prompts and watch the system adapt!

---

## Resources

- Full code: `tutorials/multi-agent-workflows/`
- Flyte docs: https://docs.flyte.org
- Questions? Join the Flyte community Slack!
- More live events: 